# Florence2 Large COCO Pruned GA P30 Mr002

This notebook was reorganized for the GitHub reproducibility package.
Original file: `GA-I_P30_MR0.02/Florence-2_Pruned[1,2,4,3]#L01f44c.ipynb`.

**Security note:** hard-coded Hugging Face tokens were removed. Use interactive login or environment variables instead.


# **Florence-2 Large Prune Denemesi**



> Başlangıçta sistem kötüydü ama başarılı olduk sonunda görülüyor.

In [ ]:
!pip uninstall -y transformers tokenizers huggingface_hub accelerate timm
!pip install -q \
  transformers==4.49.0 \
  tokenizers==0.21.0 \
  huggingface_hub==0.29.1 \
  accelerate==1.4.0 \
  timm==1.0.15 \
  sentencepiece \
  einops \
  pymoo \
  pycocotools \
  pycocoevalcap

In [ ]:
!apt-get install -y default-jdk -q


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Görüntüleri yerel diske kopyala — Drive darboğazını ortadan kaldırır
import os
if not os.path.exists('/content/val2014'):
    print("COCO kopyalanıyor...")
    !cp -r /content/drive/MyDrive/datasets/coco2014/val2014 /content/val2014
    print("✅ COCO kopyalandı")
else:
    print("✅ COCO zaten mevcut")

if not os.path.exists('/content/nocaps_images'):
    print("NoCaps kopyalanıyor...")
    !cp -r /content/drive/MyDrive/datasets/nocaps/images_val_hf /content/nocaps_images
    print("✅ NoCaps kopyalandı")
else:
    print("✅ NoCaps zaten mevcut")

if not os.path.exists('/content/proxy_images'):
    print("Proxy kopyalanıyor...")
    !cp -r /content/drive/MyDrive/proxy_coco_200/images /content/proxy_images
    print("✅ Proxy kopyalandı")
else:
    print("✅ Proxy zaten mevcut")

print("\n✅ Tüm veriler yerel diske kopyalandı!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!rm -rf /content/val2014
!cp -r /content/drive/MyDrive/datasets/coco2014/val2014 /content/val2014
import os
print(f"val2014: {len(os.listdir('/content/val2014'))} dosya")

In [ ]:
# ── 2. HÜCRE: Model yükle ─────────────────────────────────────────────
import torch
from transformers import AutoModelForCausalLM, AutoProcessor

MODEL_ID = "microsoft/Florence-2-large"

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    trust_remote_code=True,
    attn_implementation="sdpa"
).to("cuda").eval()

original_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model hazir | Parametre: {original_params:,}")

In [ ]:
# ── 3. HÜCRE: GA ─────────────────────────────────────────────────────
import json, os, torch, time
import numpy as np
from PIL import Image
from tqdm import tqdm
from pycocoevalcap.cider.cider import Cider
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.operators.sampling.rnd import BinaryRandomSampling
from pymoo.optimize import minimize
from pymoo.termination import get_termination
from pymoo.core.callback import Callback
import copy

TASK_PROMPT = "<CAPTION>"
DEVICE      = "cuda"
BATCH_SIZE  = 64
N_BLOCKS    = 12
MAX_PRUNE   = 6

PROXY_JSON  = "/content/drive/MyDrive/proxy_coco_200/proxy_coco_200_annotations.json"
PROXY_IMAGE = "/content/proxy_images"
RESULTS_DIR = "/content/drive/MyDrive/ga_results/florence2_fast"
os.makedirs(RESULTS_DIR, exist_ok=True)

with open(PROXY_JSON) as f:
    proxy_data = json.load(f)
proxy_images = proxy_data["images"]
print(f"✅ Proxy set: {len(proxy_images)} görüntü")

def repair(chromosome, max_prune):
    chromosome = chromosome.copy()
    zero_indices = [i for i, bit in enumerate(chromosome) if bit == 0]
    if len(zero_indices) > max_prune:
        excess = len(zero_indices) - max_prune
        restore_indices = np.random.choice(zero_indices, excess, replace=False)
        for idx in restore_indices:
            chromosome[idx] = 1
    return chromosome

def prune_model(model, chromosome):
    pruned_model = copy.deepcopy(model)
    layers = pruned_model.language_model.model.decoder.layers
    indices_to_keep = [i for i, bit in enumerate(chromosome) if bit == 1]
    pruned_model.language_model.model.decoder.layers = torch.nn.ModuleList(
        [layers[i] for i in indices_to_keep]
    )
    pruned_params   = sum(p.numel() for p in pruned_model.parameters())
    param_drop_rate = (original_params - pruned_params) / original_params
    return pruned_model, param_drop_rate

def run_inference(model, processor, proxy_images, device, batch_size=16):
    model.eval()
    results = {}
    for i in tqdm(range(0, len(proxy_images), batch_size), desc="Inference"):
        batch = proxy_images[i:i+batch_size]
        images, img_ids = [], []
        for item in batch:
            img_path = os.path.join(PROXY_IMAGE, item["filename"])
            images.append(Image.open(img_path).convert("RGB"))
            img_ids.append(item["image_id"])

        inputs = processor(
            text=[TASK_PROMPT] * len(images),
            images=images,
            return_tensors="pt",
            padding=True
        ).to(device, torch.float16)

        with torch.no_grad():
            generated_ids = model.generate(
                input_ids=inputs["input_ids"],
                pixel_values=inputs["pixel_values"],
                max_new_tokens=1024,
                do_sample=False,
                num_beams=1
            )

        generated_texts = processor.batch_decode(generated_ids, skip_special_tokens=False)

        for j, (img_id, gen_text, image) in enumerate(zip(img_ids, generated_texts, images)):
            parsed = processor.post_process_generation(
                gen_text,
                task=TASK_PROMPT,
                image_size=(image.width, image.height)
            )
            caption = parsed[TASK_PROMPT]
            results[str(img_id)] = [caption.strip()]

    return results

def compute_cider(results, proxy_images):
    gts = {}
    for item in proxy_images:
        img_id = str(item["image_id"])
        gts[img_id] = [c.strip() for c in item["captions"]]
    scorer = Cider()
    score, _ = scorer.compute_score(gts, results)
    return score

class PruningProblem(Problem):
    def __init__(self):
        super().__init__(n_var=N_BLOCKS, n_obj=2, n_ieq_constr=0,
                         xl=0, xu=1, vtype=bool)

    def _evaluate(self, X, out, *args, **kwargs):
        f1_list, f2_list = [], []
        for chromosome in X:
            chromosome = repair(chromosome.astype(int), MAX_PRUNE)
            pruned_model, param_drop_rate = prune_model(model, chromosome)
            results     = run_inference(pruned_model, processor, proxy_images, DEVICE, BATCH_SIZE)
            cider_score = compute_cider(results, proxy_images)
            f1_list.append(-param_drop_rate)
            f2_list.append(-cider_score)
            del pruned_model
            torch.cuda.empty_cache()
        out["F"] = np.column_stack([f1_list, f2_list])

class EarlyStoppingCallback(Callback):
    def __init__(self, patience=20):
        super().__init__()
        self.patience   = patience
        self.no_improve = 0
        self.best_f     = None

    def notify(self, algorithm):
        current_f    = algorithm.pop.get("F")
        current_best = np.min(current_f[:, 1])
        if self.best_f is None or current_best < self.best_f:
            self.best_f     = current_best
            self.no_improve = 0
        else:
            self.no_improve += 1
        gen = algorithm.n_gen
        print(f"Nesil {gen:3d} | En iyi CIDEr: {-current_best:.4f} | "
              f"Iyilesme yok: {self.no_improve}/{self.patience}")
        if self.no_improve >= self.patience:
            print("⛔ Erken durdurma tetiklendi!")
            algorithm.termination.force_termination = True

algorithm = NSGA2(
    pop_size=30,
    sampling=BinaryRandomSampling(),
    crossover=SBX(prob=0.9),
    mutation=BitflipMutation(prob=0.02),
    eliminate_duplicates=True
)
termination = get_termination("n_gen", 50)

start_time = time.time()
print("🚀 GA basliyor...")
print(f"   Model: {MODEL_ID} | Blok: {N_BLOCKS} | Pop: 30 | Nesil: 50 | Max budama: {MAX_PRUNE}")
print("-" * 60)

res = minimize(
    PruningProblem(), algorithm, termination,
    callback=EarlyStoppingCallback(patience=20),
    seed=42, verbose=False
)

elapsed = time.time() - start_time
print(f"✅ GA tamamlandi! Sure: {int(elapsed//3600)}s "
      f"{int((elapsed%3600)//60)}dk {int(elapsed%60)}sn")
print(f"   Pareto cephesi: {len(res.F)} cozum")

pareto_solutions = []
for i, (f, x) in enumerate(zip(res.F, res.X)):
    pareto_solutions.append({
        "index"          : i,
        "chromosome"     : x.tolist(),
        "param_drop_rate": float(-f[0]),
        "cider_score"    : float(-f[1])
    })

results_path = os.path.join(RESULTS_DIR, "pareto_solutions.json")
with open(results_path, "w") as f:
    json.dump({
        "model_id"       : MODEL_ID,
        "n_blocks"       : N_BLOCKS,
        "max_prune"      : MAX_PRUNE,
        "original_params": original_params,
        "solutions"      : pareto_solutions
    }, f, indent=2)

print(f"✅ Kaydedildi: {results_path}")
print(f"\nPareto Cephesi:")
print(f"{'Cozum':>6} {'Param Dususu':>12} {'CIDEr':>8}")
print("-" * 35)
for sol in sorted(pareto_solutions, key=lambda x: x["param_drop_rate"]):
    print(f"{sol['index']:>6} %{sol['param_drop_rate']*100:>10.1f} "
          f"{sol['cider_score']:>8.4f}")

In [ ]:
import json
import matplotlib.pyplot as plt

with open("/content/drive/MyDrive/ga_results/florence2_fast/pareto_solutions.json") as f:
    pareto_data = json.load(f)

solutions = sorted(pareto_data["solutions"], key=lambda x: x["param_drop_rate"])
pareto_x        = [s["param_drop_rate"] * 100 for s in solutions]
pareto_y        = [s["cider_score"] for s in solutions]

BASELINE        = 1.365
ORIGINAL_PARAMS = pareto_data["original_params"]

fig, ax1 = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor("#0f172a")
ax1.set_facecolor("#1e293b")
ax2 = ax1.twinx()

ax1.grid(color="#334155", linestyle="--", linewidth=0.6, alpha=0.7)

ax1.plot(pareto_x, pareto_y, color="#475569", linestyle="--", linewidth=1.2, zorder=2)
ax1.scatter(pareto_x, pareto_y, color="#38bdf8", s=80, zorder=3, label="Pareto noktaları (proxy, 200 görüntü)")

for s in solutions:
    ax1.annotate(
        f"%{s['param_drop_rate']*100:.1f}\n{s['cider_score']:.4f}",
        (s["param_drop_rate"] * 100, s["cider_score"]),
        textcoords="offset points", xytext=(0, 12), ha="center",
        color="#38bdf8", fontsize=8,
        bbox=dict(boxstyle="round,pad=0.2", facecolor="#0f172a", edgecolor="#38bdf8", alpha=0.8)
    )

ax1.axhline(y=BASELINE, color="#f59e0b", linestyle="--", linewidth=1.2, alpha=0.7)
ax1.text(12.5, BASELINE + 0.01, f"Baseline: {BASELINE}", color="#f59e0b", fontsize=9, ha="right")

remaining = [ORIGINAL_PARAMS * (1 - s["param_drop_rate"]) / 1e9 for s in solutions]
ax2.plot(pareto_x, remaining, color="#a78bfa", linestyle="-", linewidth=1.5, zorder=6, label="Kalan parametre (B)")
ax2.scatter(pareto_x, remaining, color="#a78bfa", s=45, zorder=7)

ax1.set_xlim(2, 15)
ax1.set_ylim(0, 1.5)
ax1.set_xlabel("Parametre Düşüşü (%)", color="#94a3b8", fontsize=12)
ax1.set_ylabel("CIDEr Skoru (Proxy)", color="#94a3b8", fontsize=12)
ax1.tick_params(colors="#94a3b8")
for spine in ax1.spines.values():
    spine.set_edgecolor("#334155")

ax2.set_ylabel("Kalan Parametre Sayısı (Milyar)", color="#a78bfa", fontsize=11)
ax2.tick_params(colors="#a78bfa")
for spine in ax2.spines.values():
    spine.set_edgecolor("#334155")
ax2.spines["right"].set_edgecolor("#a78bfa")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2,
           facecolor="#1e293b", edgecolor="#334155", labelcolor="#94a3b8", fontsize=9, loc="upper right")

ax1.set_title("Florence-2 — Pareto Cephesi (Proxy, 200 Görüntü)", color="#f1f5f9", fontsize=13, pad=14)

plt.tight_layout()
plt.savefig("/content/drive/MyDrive/ga_results/florence2_fast/florence2_pareto_proxy.png",
            dpi=150, bbox_inches="tight", facecolor="#0f172a")
plt.show()
print("✅ Kaydedildi")

In [ ]:
# Session Yenilenmiş Base Model tekrardan ekleyelim
import os, json, torch, copy, time
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM
from pycocotools.coco import COCO
from pycocoevalcap.eval import COCOEvalCap
from google.colab import drive

drive.mount('/content/drive')

DEVICE      = "cuda"
BATCH_SIZE  = 64
MODEL_ID    = "microsoft/Florence-2-large"
PARETO_JSON = "/content/drive/MyDrive/ga_results/florence2_fast/pareto_solutions.json"
SELECTED    = [1, 2, 4, 3]
label_map   = {1: "low", 2: "mid", 4: "high", 3: "extra_high"}
TASK_PROMPT = "<CAPTION>"

with open(PARETO_JSON) as f:
    pareto_data = json.load(f)
selected_solutions = [s for s in pareto_data["solutions"] if s["index"] in SELECTED]
selected_solutions.sort(key=lambda x: x["param_drop_rate"])

print("Base model yükleniyor...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    trust_remote_code=True,
    attn_implementation="sdpa"
).to(DEVICE)
base_model.eval()
print(f"✅ Base model hazır")


def prune_model(base_model, chromosome):
    model = copy.deepcopy(base_model)
    layers = model.language_model.model.decoder.layers
    keep_indices = [i for i, keep in enumerate(chromosome) if keep]
    new_layers = torch.nn.ModuleList([layers[i] for i in keep_indices])
    for layer in new_layers:
        for param in layer.parameters():
            if not param.is_contiguous():
                param.data = param.data.contiguous()
    model.language_model.model.decoder.layers = new_layers
    return model


def run_inference_florence(model, processor, images, img_dir, id_fn, device, batch_size=64):
    """Florence-2 encoder-decoder — trim fix YOK"""
    results = []
    for i in range(0, len(images), batch_size):
        batch = images[i:i+batch_size]
        pil_imgs, img_ids = [], []
        for img_info in batch:
            fname, img_id = id_fn(img_info)
            pil_imgs.append(Image.open(os.path.join(img_dir, fname)).convert("RGB"))
            img_ids.append(img_id)

        inputs = processor(
            images=pil_imgs,
            text=[TASK_PROMPT] * len(pil_imgs),
            return_tensors="pt",
            padding=True
        ).to(device, torch.float16)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=50,
                num_beams=1,
                do_sample=False
            )

        # Florence-2 encoder-decoder — direkt decode, trim yok
        raw_captions = processor.batch_decode(out, skip_special_tokens=True)
        captions = [
            processor.post_process_generation(
                c, task=TASK_PROMPT, image_size=(1, 1)
            ).get("<CAPTION>", c).strip()
            for c in raw_captions
        ]

        for img_id, cap in zip(img_ids, captions):
            results.append({"image_id": img_id, "caption": cap})

        if (i // batch_size) % 50 == 0:
            print(f"  {i+len(batch)}/{len(images)}")
            print(f"  Örnek caption: {captions[0]}")

    return results


def compute_metrics(results, gt_json, save_dir):
    res_path = os.path.join(save_dir, "_tmp_results.json")
    with open(res_path, "w") as f:
        json.dump(results, f)
    coco_gt   = COCO(gt_json)
    coco_res  = coco_gt.loadRes(res_path)
    evaluator = COCOEvalCap(coco_gt, coco_res)
    try:
        evaluator.evaluate()
    except Exception as e:
        print(f"  ⚠️ SPICE hatası (görmezden gelindi): {e}")
    return {k: v for k, v in evaluator.eval.items() if k in ["CIDEr", "Bleu_4"]}


print("✅ Tüm fonksiyonlar hazır")

# **Final Evaluation w/ COCO 5K Karpathy Test**

In [ ]:
!apt-get install -y default-jdk -q
!pip install pycocotools pycocoevalcap -q

In [ ]:
import os, json, shutil, time, torch, copy
from PIL import Image
from pycocotools.coco import COCO
from pycocoevalcap.eval import COCOEvalCap

# ── Görüntüleri diske kopyala (yoksa) ──
COCO_DST = "/content/images/coco_val2014"
COCO_SRC = "/content/drive/MyDrive/datasets/coco2014/val2014"
TEST_JSON = "/content/drive/MyDrive/coco_karpathy/coco_karpathy_test.json"
os.makedirs(COCO_DST, exist_ok=True)

with open(TEST_JSON) as f:
    test_images = json.load(f)

if len(os.listdir(COCO_DST)) < len(test_images):
    print(f"5000 görüntü kopyalanıyor...")
    t0 = time.time()
    for item in test_images:
        fname = item["image"].split("/")[-1]
        src = os.path.join(COCO_SRC, fname)
        dst = os.path.join(COCO_DST, fname)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
    print(f"✅ Kopyalandı — {time.time()-t0:.0f} sn")
else:
    print("✅ Görüntüler zaten diskte")

# ── Eval ──
IMG_DIR  = COCO_DST
GT_JSON  = "/content/drive/MyDrive/coco_karpathy/coco_karpathy_test_gt.json"
SAVE_DIR = "/content/drive/MyDrive/ga_results/florence2_fast/final_eval_2"
os.makedirs(SAVE_DIR, exist_ok=True)

TASK_PROMPT = "<CAPTION>"
print(f"COCO test seti: {len(test_images)} görüntü")

def coco_id_fn(img_info):
    fname  = img_info["image"].split("/")[-1]
    img_id = int(fname.split("_")[-1].split(".")[0])
    return fname, img_id

def run_inference_florence_coco(model, processor, images, img_dir, id_fn, device, batch_size=64):
    results = []
    for i in range(0, len(images), batch_size):
        batch = images[i:i+batch_size]
        pil_imgs, img_ids = [], []
        for img_info in batch:
            fname, img_id = id_fn(img_info)
            pil_imgs.append(Image.open(os.path.join(img_dir, fname)).convert("RGB"))
            img_ids.append(img_id)

        inputs = processor(
            images=pil_imgs,
            text=[TASK_PROMPT] * len(pil_imgs),
            return_tensors="pt",
            padding=True
        ).to(device, torch.float16)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=50,
                num_beams=1,
                do_sample=False
            )

        # Florence-2 encoder-decoder — TRIM YOK
        raw_captions = processor.batch_decode(out, skip_special_tokens=True)
        captions = [
            processor.post_process_generation(
                c, task=TASK_PROMPT, image_size=(1, 1)
            ).get("<CAPTION>", c).strip()
            for c in raw_captions
        ]

        for img_id, cap in zip(img_ids, captions):
            results.append({"image_id": img_id, "caption": cap})

        if (i // batch_size) % 50 == 0:
            print(f"  {i+len(batch)}/{len(images)}")
            print(f"  Örnek caption: {captions[0]}")

    return results

def compute_metrics(results, gt_json, save_dir):
    res_path = os.path.join(save_dir, "_tmp_results.json")
    with open(res_path, "w") as f:
        json.dump(results, f)
    coco_gt   = COCO(gt_json)
    coco_res  = coco_gt.loadRes(res_path)
    evaluator = COCOEvalCap(coco_gt, coco_res)
    try:
        evaluator.evaluate()
    except Exception as e:
        print(f"  ⚠️ SPICE hatası (görmezden gelindi): {e}")
    return {k: v for k, v in evaluator.eval.items() if k in ["CIDEr", "Bleu_4"]}

summary_coco = []
t_total = time.time()

for sol in selected_solutions:
    label = label_map[sol["index"]]
    print(f"\n{'='*50}")
    print(f"[{label.upper()}] index:{sol['index']} | %{sol['param_drop_rate']*100:.1f} düşüş")

    model = prune_model(base_model, sol["chromosome"])
    n_kept = sum(sol["chromosome"])
    print(f"  {len(sol['chromosome']) - n_kept} blok silindi, {n_kept} kaldı")

    t0 = time.time()
    results = run_inference_florence_coco(
        model, processor, test_images,
        IMG_DIR, coco_id_fn, DEVICE, BATCH_SIZE
    )
    metrics = compute_metrics(results, GT_JSON, SAVE_DIR)

    cider = metrics.get("CIDEr", 0)
    bleu4 = metrics.get("Bleu_4", 0)
    print(f"  CIDEr : {cider:.4f}")
    print(f"  BLEU-4: {bleu4:.4f}")
    print(f"  Süre  : {(time.time()-t0)/60:.1f} dk")

    entry = {
        "label": label,
        "index": sol["index"],
        "param_drop_rate": sol["param_drop_rate"],
        "proxy_cider": sol["cider_score"],
        "final_cider": cider,
        "final_bleu4": bleu4,
        "chromosome": sol["chromosome"]
    }
    summary_coco.append(entry)

    out_path = os.path.join(SAVE_DIR, f"florence2_{label}_eval_2.json")
    with open(out_path, "w") as f:
        json.dump(entry, f, indent=2)
    print(f"  ✅ Kaydedildi: {out_path}")

    del model
    torch.cuda.empty_cache()

summary_path = os.path.join(SAVE_DIR, "florence2_final_summary_2.json")
with open(summary_path, "w") as f:
    json.dump(summary_coco, f, indent=2)

print(f"\n{'='*50}")
print(f"✅ COCO tamamlandı — toplam süre: {(time.time()-t_total)/60:.1f} dk")
print(f"\nÖZET:")
print(f"{'Label':<12} {'Param Düşüş':>12}  {'Proxy CIDEr':>12}  {'Final CIDEr':>12}  {'BLEU-4':>8}")
print("-" * 62)
for s in summary_coco:
    print(f"{s['label']:<12} %{s['param_drop_rate']*100:>10.1f}  "
          f"{s['proxy_cider']:>12.4f}  {s['final_cider']:>12.4f}  {s['final_bleu4']:>8.4f}")

# **Final Evaluation w/ NoCaps 4.5K Karpathy Test**

In [ ]:
# ── Florence-2 NoCaps Final Eval (DÜZELTİLMİŞ) ──────────────

IMG_DIR_NOCAPS  = NOCAPS_DST
GT_JSON_NOCAPS  = "/content/drive/MyDrive/datasets/nocaps/nocaps_val_4500_captions_domain_norm.json"
SAVE_DIR_NOCAPS = "/content/drive/MyDrive/ga_results/florence2_fast/final_eval_nocaps"
os.makedirs(SAVE_DIR_NOCAPS, exist_ok=True)

with open(GT_JSON_NOCAPS) as f:
    nocaps_data = json.load(f)
nocaps_images = nocaps_data["images"]
print(f"NoCaps val seti: {len(nocaps_images)} görüntü")

TASK_PROMPT = "<CAPTION>"

def nocaps_id_fn(img_info):
    return img_info["file_name"], img_info["id"]

def run_inference_florence(model, processor, images, img_dir, id_fn, device, batch_size=64):
    results = []
    for i in range(0, len(images), batch_size):
        batch = images[i:i+batch_size]
        pil_imgs, img_ids = [], []
        for img_info in batch:
            fname, img_id = id_fn(img_info)
            pil_imgs.append(Image.open(os.path.join(img_dir, fname)).convert("RGB"))
            img_ids.append(img_id)

        inputs = processor(
            images=pil_imgs,
            text=[TASK_PROMPT] * len(pil_imgs),
            return_tensors="pt",
            padding=True
        ).to(device, torch.float16)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=50,
                num_beams=1,
                do_sample=False
            )

        # Florence-2 encoder-decoder — TRIM YOK
        raw_captions = processor.batch_decode(out, skip_special_tokens=True)
        captions = [
            processor.post_process_generation(
                c, task=TASK_PROMPT, image_size=(1, 1)
            ).get("<CAPTION>", c).strip()
            for c in raw_captions
        ]

        for img_id, cap in zip(img_ids, captions):
            results.append({"image_id": img_id, "caption": cap})

        if (i // batch_size) % 50 == 0:
            print(f"  {i+len(batch)}/{len(images)}")
            print(f"  Örnek caption: {captions[0]}")

    return results

def compute_metrics(results, gt_json, save_dir):
    res_path = os.path.join(save_dir, "_tmp_results.json")
    with open(res_path, "w") as f:
        json.dump(results, f)
    coco_gt   = COCO(gt_json)
    coco_res  = coco_gt.loadRes(res_path)
    evaluator = COCOEvalCap(coco_gt, coco_res)
    try:
        evaluator.evaluate()
    except Exception as e:
        print(f"  ⚠️ SPICE hatası (görmezden gelindi): {e}")
    return {k: v for k, v in evaluator.eval.items() if k in ["CIDEr", "Bleu_4"]}

summary_nocaps = []
t_total = time.time()

for sol in selected_solutions:
    label = label_map[sol["index"]]
    print(f"\n{'='*50}")
    print(f"[{label.upper()}] index:{sol['index']} | %{sol['param_drop_rate']*100:.1f} düşüş")

    model = prune_model(base_model, sol["chromosome"])
    n_kept = sum(sol["chromosome"])
    print(f"  {len(sol['chromosome']) - n_kept} blok silindi, {n_kept} kaldı")

    t0 = time.time()
    results = run_inference_florence(
        model, processor, nocaps_images,
        IMG_DIR_NOCAPS, nocaps_id_fn, DEVICE, BATCH_SIZE
    )
    metrics = compute_metrics(results, GT_JSON_NOCAPS, SAVE_DIR_NOCAPS)

    cider = metrics.get("CIDEr", 0)
    bleu4 = metrics.get("Bleu_4", 0)
    print(f"  CIDEr : {cider:.4f}")
    print(f"  BLEU-4: {bleu4:.4f}")
    print(f"  Süre  : {(time.time()-t0)/60:.1f} dk")

    entry = {
        "label": label,
        "index": sol["index"],
        "param_drop_rate": sol["param_drop_rate"],
        "proxy_cider": sol["cider_score"],
        "final_cider": cider,
        "final_bleu4": bleu4,
        "chromosome": sol["chromosome"]
    }
    summary_nocaps.append(entry)

    out_path = os.path.join(SAVE_DIR_NOCAPS, f"florence2_nocaps_{label}_eval.json")
    with open(out_path, "w") as f:
        json.dump(entry, f, indent=2)
    print(f"  ✅ Kaydedildi: {out_path}")

    del model
    torch.cuda.empty_cache()

summary_path = os.path.join(SAVE_DIR_NOCAPS, "florence2_nocaps_summary.json")
with open(summary_path, "w") as f:
    json.dump(summary_nocaps, f, indent=2)

print(f"\n{'='*50}")
print(f"✅ NoCaps tamamlandı — toplam süre: {(time.time()-t_total)/60:.1f} dk")
print(f"\nÖZET:")
print(f"{'Label':<12} {'Param Düşüş':>12}  {'Proxy CIDEr':>12}  {'Final CIDEr':>12}  {'BLEU-4':>8}")
print("-" * 62)
for s in summary_nocaps:
    print(f"{s['label']:<12} %{s['param_drop_rate']*100:>10.1f}  "
          f"{s['proxy_cider']:>12.4f}  {s['final_cider']:>12.4f}  {s['final_bleu4']:>8.4f}")